# HTRU2 Pulsar — 纯 KAN 分类器 (基准)
## Parkes 64m 射电望远镜 → 脉冲星候选体分类 → 8 特征输入 → KAN [8,64,64,1]
无 PDE、无量子，作为 KAN+PDE+量子方案的对比基准

In [1]:
import numpy as np
import torch, torch.nn as nn
import pandas as pd, warnings, json, os, time, io, zipfile, requests
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUTPUT_DIR = 'D:/QPDE/photo+kan/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'设备: {DEVICE}')

设备: cuda


### 1. RBF-KAN 定义

In [2]:
class KANLayer(nn.Module):
    def __init__(self, in_dim, out_dim, grid_size=8):
        super().__init__()
        self.register_buffer('centers', torch.linspace(-1, 1, grid_size))
        self.width = nn.Parameter(torch.ones(1) * 2.0 / grid_size)
        self.coef = nn.Parameter(torch.randn(out_dim, in_dim, grid_size) * 0.1)
        self.base = nn.Linear(in_dim, out_dim)
    def forward(self, x):
        x_exp = x.unsqueeze(-1); c = self.centers.view(1, 1, -1)
        rbf = torch.exp(-((x_exp - c) / self.width)**2)
        return torch.einsum('big,oig->bo', rbf, self.coef) + self.base(x)

class KAN(nn.Module):
    def __init__(self, layers, grid_size=8):
        super().__init__()
        self.layers = nn.ModuleList([KANLayer(layers[i], layers[i+1], grid_size) for i in range(len(layers)-1)])
    def forward(self, x):
        for l in self.layers[:-1]: x = torch.tanh(l(x))
        return self.layers[-1](x)

print('KAN 类定义完成')

KAN 类定义完成


### 2. 加载 HTRU2 脉冲星数据

In [3]:
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00372/HTRU2.zip'
zf = zipfile.ZipFile(io.BytesIO(requests.get(url).content))
df = pd.read_csv(zf.open('HTRU_2.csv'), header=None,
    names=['mean_ip','std_ip','excess_kurtosis_ip','skewness_ip',
           'mean_dm','std_dm','excess_kurtosis_dm','skewness_dm','class'])
X = df.drop(columns=['class']).values.astype(float)
y = np.where(df['class'].values == 1, 1, -1)  # pulsar=+1, RFI=-1
print(f'HTRU2: {len(y)} 条, 脉冲星={(y==1).sum()}, RFI={(y==-1).sum()}')

HTRU2: 17898 条, 脉冲星=1639, RFI=16259


In [4]:
# 采样 + 标准化
rng = np.random.default_rng(42)
idx = rng.choice(len(y), 2000, replace=False)
X_s, y_s = X[idx], y[idx]
X_s = StandardScaler().fit_transform(X_s)

# 训练/测试划分
n_train = 1600
X_tr, y_tr = X_s[:n_train], y_s[:n_train]
X_te, y_te = X_s[n_train:], y_s[n_train:]
print(f'训练: {n_train} 条, 测试: {len(y_te)} 条')

训练: 1600 条, 测试: 400 条


In [5]:
X_tr_t = torch.tensor(X_tr, dtype=torch.float32, device=DEVICE)
y_tr_t = torch.tensor(y_tr, dtype=torch.float32, device=DEVICE)
X_te_t = torch.tensor(X_te, dtype=torch.float32, device=DEVICE)
y_te_t = torch.tensor(y_te, dtype=torch.float32, device=DEVICE)
print('Tensor 转换完成')

Tensor 转换完成


### 3. 训练 KAN 分类器

In [6]:
model = KAN([8, 64, 64, 1], grid_size=8).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
N_epoch = 500

hist = {'loss': [], 'acc': [], 'rmse': []}
t0 = time.time()
for epoch in range(N_epoch):
    model.train()
    pred = model(X_tr_t).squeeze()
    loss = torch.tanh(pred * y_tr_t).mean()  # tanh margin loss
    opt.zero_grad(); loss.backward(); opt.step()
    hist['loss'].append(float(loss.item()))

    if epoch % 100 == 0:
        model.eval()
        with torch.no_grad():
            pred_te = model(X_te_t).squeeze()
            acc = ((pred_te * y_te_t) > 0).float().mean().item()
            rmse = (pred_te - y_te_t).pow(2).mean().sqrt().item()
            hist['acc'].append(float(acc))
            hist['rmse'].append(float(rmse))
        model.train()
        print(f'epoch {epoch}: loss={loss.item():.4e}, test_acc={acc:.4f}, test_rmse={rmse:.4f}')

elapsed = time.time() - t0
print(f'训练完成: {elapsed:.1f}s')

epoch 0: loss=7.1336e-01, test_acc=0.7950, test_rmse=1.0407
epoch 100: loss=-9.6920e-01, test_acc=0.0325, test_rmse=5.3170
epoch 200: loss=-9.7314e-01, test_acc=0.0300, test_rmse=5.6232
epoch 300: loss=-9.7450e-01, test_acc=0.0300, test_rmse=5.8902
epoch 400: loss=-9.7468e-01, test_acc=0.0300, test_rmse=6.1107
训练完成: 4.0s


### 4. 评估 & 保存结果

In [7]:
model.eval()
with torch.no_grad():
    pred_te = model(X_te_t).squeeze().cpu().numpy()
    acc = np.mean((pred_te * y_te) > 0)
    rmse = np.sqrt(np.mean((pred_te - y_te)**2))
    pred_tr = model(X_tr_t).squeeze().cpu().numpy()
    tr_acc = np.mean((pred_tr * y_tr) > 0)

# 按类别统计准确率
pulsar_mask = y_te == 1
rfi_mask = y_te == -1
pulsar_acc = np.mean((pred_te[pulsar_mask] * y_te[pulsar_mask]) > 0) if pulsar_mask.sum() > 0 else 0
rfi_acc = np.mean((pred_te[rfi_mask] * y_te[rfi_mask]) > 0) if rfi_mask.sum() > 0 else 0

print(f'纯KAN: test_acc={acc:.4f}, RMSE={rmse:.4f}, 耗时={elapsed:.1f}s')
print(f'  pulsar_acc={pulsar_acc:.4f}, RFI_acc={rfi_acc:.4f}, train_acc={tr_acc:.4f}')

纯KAN: test_acc=0.0300, RMSE=6.2740, 耗时=4.0s
  pulsar_acc=0.2174, RFI_acc=0.0056, train_acc=0.0125


In [8]:
# 保存 .npy 文件
np.save(f'{OUTPUT_DIR}/htru2_kan_sa_pred_test.npy', pred_te)
np.save(f'{OUTPUT_DIR}/htru2_kan_sa_pred_train.npy', pred_tr)
np.save(f'{OUTPUT_DIR}/htru2_kan_sa_X_train.npy', X_tr)
np.save(f'{OUTPUT_DIR}/htru2_kan_sa_y_train.npy', y_tr)
np.save(f'{OUTPUT_DIR}/htru2_kan_sa_X_test.npy', X_te)
np.save(f'{OUTPUT_DIR}/htru2_kan_sa_y_test.npy', y_te)

res = {
    'method': 'kan_standalone',
    'dataset': 'HTRU2_pulsar',
    'telescope': 'Parkes_64m_radio',
    'n_train': n_train, 'n_test': len(y_te),
    'epochs': N_epoch, 'time_sec': float(elapsed),
    'test_acc': float(acc), 'test_rmse': float(rmse),
    'train_acc': float(tr_acc),
    'pulsar_accuracy': float(pulsar_acc),
    'rfi_accuracy': float(rfi_acc),
    'loss_history': hist['loss'],
    'acc_history': hist['acc'],
    'rmse_history': hist['rmse'],
}
with open(f'{OUTPUT_DIR}/result_kan_standalone.json','w') as f:
    json.dump(res, f, indent=2)
print('所有结果已保存')

所有结果已保存
